In [2]:
!pip install lightgbm


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error
import lightgbm as lgb
import joblib

df = pd.read_csv('cdr_dataset_1M.csv')

# Feature Engineering
# Create high_cost_flag (top 5% calls)
threshold = df['call_cost_usd'].quantile(0.95)
df['high_cost_flag'] = (df['call_cost_usd'] > threshold).astype(int)

# Encode categorical features
le_call_type = LabelEncoder()
df['call_type_enc'] = le_call_type.fit_transform(df['call_type'])

le_tower = LabelEncoder()
df['tower_id_enc'] = le_tower.fit_transform(df['tower_id'])

# Time features (cyclical encoding)
df['hour_sin'] = np.sin(2 * np.pi * df['call_hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['call_hour']/24)

# Features & Targets
features = ['call_duration_seconds', 'call_type_enc', 'tower_id_enc', 'hour_sin', 'hour_cos']
X_class = df[features]
y_class = df['high_cost_flag']

X_reg = df[features]
y_reg = df['call_cost_usd']

# Train/Test Split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_class, y_class, test_size=0.2, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# LightGBM Classifier (High-Cost Call)

train_data = lgb.Dataset(Xc_train, label=yc_train)
test_data = lgb.Dataset(Xc_test, label=yc_test, reference=train_data)

params_cls = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'verbose': -1
}
clf_model = lgb.train(
    params_cls,
    train_data,
    num_boost_round=500,
    valid_sets=[test_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

yc_pred_prob = clf_model.predict(Xc_test)
yc_pred = (yc_pred_prob > 0.5).astype(int)

# Metrics
print("----- Classification Report -----")
print(classification_report(yc_test, yc_pred))
print("AUC Score:", roc_auc_score(yc_test, yc_pred_prob))

# Save Classifier
clf_model.save_model('./models/lgbm_classifier.txt')

# LightGBM Regressor (Call Cost)

train_data_reg = lgb.Dataset(Xr_train, label=yr_train)
test_data_reg = lgb.Dataset(Xr_test, label=yr_test, reference=train_data_reg)

params_reg = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'verbose': -1
}

reg_model = lgb.train(
    params_reg,
    train_data_reg,
    num_boost_round=500,
    valid_sets=[test_data_reg],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

yr_pred = reg_model.predict(Xr_test)

rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
print("----- Regression RMSE -----")
print(rmse)

# Save Regressor
reg_model.save_model('./models/lgbm_regressor.txt')


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[40]	valid_0's auc: 0.991264
----- Classification Report -----
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    190024
           1       0.76      0.84      0.80      9976

    accuracy                           0.98    200000
   macro avg       0.88      0.91      0.90    200000
weighted avg       0.98      0.98      0.98    200000

AUC Score: 0.9912642212652935
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[40]	valid_0's rmse: 0.518044
----- Regression RMSE -----
0.5180436941243649
